# Preprocesamiento y clustering K-Means

Este notebook prepara los registros de `data/raw` (los mismos archivos que se consolidan en `0_dataprocessed.ipynb`) usando `provincia`, `Canton`, `Parroquia`, `Servicio` y `Subtipo`. Las variables son categóricas, por lo que se codifican con one-hot encoding disperso; no se aplica escalado numérico.

Para limitar el uso de memoria, se leen los CSV por bloques y se agrupan combinaciones idénticas antes del clustering. La frecuencia de cada combinación se usa como peso, preservando la contribución de los registros repetidos. Se usa `MiniBatchKMeans`, una variante de K-Means adecuada para conjuntos grandes.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import OneHotEncoder
from IPython.display import display

# Ajusta este valor para probar una cantidad distinta de grupos.
K = 5
TAMANO_BLOQUE = 250_000
VARIABLES = ["provincia", "canton", "parroquia", "servicio", "subtipo"]

# Acepta tanto data/raw como raw según el directorio de inicio del kernel.
base = Path.cwd().resolve()
candidatas = [
    ruta
    for directorio in (base, *base.parents)
    for ruta in (directorio / "data" / "raw", directorio / "raw")
]
carpeta_raw = next((ruta for ruta in candidatas if ruta.is_dir()), None)
if carpeta_raw is None:
    raise FileNotFoundError(f"No se encontró data/raw desde {base}")

archivos = sorted(
    archivo for archivo in carpeta_raw.rglob("*")
    if archivo.is_file() and archivo.suffix.lower() in {".csv", ".xlsx"}
)
if not archivos:
    raise FileNotFoundError(f"No se encontraron CSV o XLSX en {carpeta_raw}")

print(f"Carpeta de datos: {carpeta_raw}")
print(f"Archivos encontrados: {len(archivos)}")

Carpeta de datos: C:\Users\PAULA\Documents\IntegradorG3\data\raw
Archivos encontrados: 61


## 1. Lectura y agrupación de registros

Se reconocen los encabezados sin distinguir mayúsculas/minúsculas (por ejemplo, `Canton` y `canton`). Los valores vacíos se convierten en `(sin dato)`. Cada archivo CSV se procesa en bloques; los XLSX se leen desde su primera hoja, igual que en la carga anterior.

In [2]:
# Mapea encabezados originales a los nombres canónicos solicitados.
MAPA_COLUMNAS = {variable.casefold(): variable for variable in VARIABLES}
conteos_parciales = []

def normalizar_y_contar(bloque: pd.DataFrame, archivo: Path) -> pd.DataFrame:
    """Normaliza las cinco categorías y cuenta cada combinación del bloque."""
    renombres = {
        columna: MAPA_COLUMNAS[str(columna).strip().casefold()]
        for columna in bloque.columns
        if str(columna).strip().casefold() in MAPA_COLUMNAS
    }
    bloque = bloque.rename(columns=renombres)
    faltantes = [variable for variable in VARIABLES if variable not in bloque.columns]
    if faltantes:
        raise ValueError(f"{archivo.name}: faltan las columnas {faltantes}")

    bloque = bloque[VARIABLES].copy()
    for variable in VARIABLES:
        bloque[variable] = (
            bloque[variable]
            .fillna("(sin dato)")
            .astype(str)
            .str.strip()
            .replace("", "(sin dato)")
        )
    return (
        bloque.groupby(VARIABLES, as_index=False, dropna=False)
        .size()
        .rename(columns={"size": "frecuencia"})
    )

for indice, archivo in enumerate(archivos, start=1):
    print(f"[{indice}/{len(archivos)}] Procesando {archivo.relative_to(carpeta_raw)}")
    usar_columna = lambda nombre: str(nombre).strip().casefold() in MAPA_COLUMNAS

    if archivo.suffix.lower() == ".csv":
        lector = pd.read_csv(
            archivo, sep=";", usecols=usar_columna, chunksize=TAMANO_BLOQUE
        )
        for bloque in lector:
            conteos_parciales.append(normalizar_y_contar(bloque, archivo))
    else:
        bloque = pd.read_excel(archivo, usecols=usar_columna)
        conteos_parciales.append(normalizar_y_contar(bloque, archivo))

if not conteos_parciales:
    raise ValueError("No se obtuvieron registros de los archivos encontrados.")

# Consolida conteos parciales: combinaciones repetidas conservan su frecuencia total.
datos_agrupados = (
    pd.concat(conteos_parciales, ignore_index=True)
    .groupby(VARIABLES, as_index=False, dropna=False)["frecuencia"]
    .sum()
)
total_registros = int(datos_agrupados["frecuencia"].sum())
print(f"Registros procesados: {total_registros:,}")
print(f"Combinaciones únicas de las variables: {len(datos_agrupados):,}")
display(datos_agrupados.head())

[1/61] Procesando data2021\emergencias_agosto_2021.csv
[2/61] Procesando data2021\emergencias_diciembre_2021.csv
[3/61] Procesando data2021\emergencias_julio_2021.csv
[4/61] Procesando data2021\emergencias_noviembre_2021.csv
[5/61] Procesando data2021\emergencias_octubre_2021.csv
[6/61] Procesando data2021\emergencias_septiembre_2021.csv
[7/61] Procesando data2022\emergencias_abril_2022.csv
[8/61] Procesando data2022\emergencias_agosto_2022.csv
[9/61] Procesando data2022\emergencias_diciembre_2022.csv
[10/61] Procesando data2022\emergencias_enero_2022.csv
[11/61] Procesando data2022\emergencias_febrero_2022.csv
[12/61] Procesando data2022\emergencias_julio_2022.csv
[13/61] Procesando data2022\emergencias_junio_2022.csv
[14/61] Procesando data2022\emergencias_marzo_2022 (1).csv
[15/61] Procesando data2022\emergencias_mayo_2022.csv
[16/61] Procesando data2022\emergencias_noviembre_2022 (1).csv
[17/61] Procesando data2022\emergencias_octubre_2022.csv
[18/61] Procesando data2022\emergencia

,provincia,canton,parroquia,servicio,subtipo,frecuencia
0,(sin dato),(sin dato),(sin dato),(sin dato),(sin dato),7
1,(sin dato),(sin dato),(sin dato),Gestión Sanitaria,Accidente de tránsito,4
2,(sin dato),(sin dato),(sin dato),Gestión Sanitaria,Alcohol,5
3,(sin dato),(sin dato),(sin dato),Gestión Sanitaria,Alteración consciencia / lipotimia,16
4,(sin dato),(sin dato),(sin dato),Gestión Sanitaria,Alza térmica - fiebre,2


## 2. Codificación y clustering

One-hot encoding representa cada categoría como una columna binaria y mantiene la matriz en formato disperso. K-Means agrupa combinaciones similares; `frecuencia` se usa como peso para que una combinación presente muchas veces cuente proporcionalmente más. Si existen menos combinaciones únicas que `K`, el número de clusters se reduce automáticamente.

In [3]:
if K < 2:
    raise ValueError("K debe ser al menos 2.")
if len(datos_agrupados) < 2:
    raise ValueError("Se necesitan al menos dos combinaciones distintas para aplicar K-Means.")

# Compatibilidad entre versiones de scikit-learn.
try:
    codificador = OneHotEncoder(
        handle_unknown="ignore", sparse_output=True, dtype=np.float32
    )
except TypeError:
    codificador = OneHotEncoder(
        handle_unknown="ignore", sparse=True, dtype=np.float32
    )

X = codificador.fit_transform(datos_agrupados[VARIABLES])
k_real = min(K, len(datos_agrupados))
modelo_kmeans = MiniBatchKMeans(
    n_clusters=k_real,
    random_state=42,
    n_init=10,
    batch_size=4096,
)
etiquetas_cluster = modelo_kmeans.fit_predict(
    X, sample_weight=datos_agrupados["frecuencia"].to_numpy()
)

datos_preprocesados = datos_agrupados.copy()
datos_preprocesados["cluster"] = etiquetas_cluster
print(f"Dimensiones de la matriz codificada: {X.shape}")
print(f"Clusters generados: {k_real}")
display(datos_preprocesados.head())

Dimensiones de la matriz codificada: (166886, 1934)
Clusters generados: 5


,provincia,canton,parroquia,servicio,subtipo,frecuencia,cluster
0,(sin dato),(sin dato),(sin dato),(sin dato),(sin dato),7,2
1,(sin dato),(sin dato),(sin dato),Gestión Sanitaria,Accidente de tránsito,4,2
2,(sin dato),(sin dato),(sin dato),Gestión Sanitaria,Alcohol,5,2
3,(sin dato),(sin dato),(sin dato),Gestión Sanitaria,Alteración consciencia / lipotimia,16,2
4,(sin dato),(sin dato),(sin dato),Gestión Sanitaria,Alza térmica - fiebre,2,2


In [4]:
resumen_clusters = (
    datos_preprocesados.groupby("cluster", as_index=False)
    .agg(
        registros=("frecuencia", "sum"),
        combinaciones=("frecuencia", "size"),
    )
    .sort_values("cluster")
)
print("Registros y combinaciones asignados a cada cluster:")
display(resumen_clusters)

# Muestra las combinaciones más frecuentes dentro de cada grupo.
combinaciones_top = (
    datos_preprocesados.sort_values("frecuencia", ascending=False)
    .groupby("cluster", group_keys=False)
    .head(5)
    .sort_values(["cluster", "frecuencia"], ascending=[True, False])
)
display(combinaciones_top)

Registros y combinaciones asignados a cada cluster:


,cluster,registros,combinaciones
0,0,3788670,8758
1,1,6869369,62044
2,2,2889110,89529
3,3,3000270,1303
4,4,833183,5252


,provincia,canton,parroquia,servicio,subtipo,frecuencia,cluster
138451,PICHINCHA,QUITO,"QUITO DISTRITO METROPOLITANO, CABECERA CANTONA...",Seguridad Ciudadana,Patrullaje policial en el sector solicitado,463454,0
138464,PICHINCHA,QUITO,"QUITO DISTRITO METROPOLITANO, CABECERA CANTONA...",Seguridad Ciudadana,Presencia policial,338610,0
138433,PICHINCHA,QUITO,"QUITO DISTRITO METROPOLITANO, CABECERA CANTONA...",Seguridad Ciudadana,Libadores,216411,0
138395,PICHINCHA,QUITO,"QUITO DISTRITO METROPOLITANO, CABECERA CANTONA...",Seguridad Ciudadana,Escándalo en espacio público,125510,0
138611,PICHINCHA,QUITO,"QUITO DISTRITO METROPOLITANO, CABECERA CANTONA...",Servicios Municipales,Apoyo institucional,85734,0
146820,SANTO DOMINGO DE LOS TSACHILAS,SANTO DOMINGO,"SANTO DOMINGO DE LOS COLORADOS, CABECERA CANTO...",Seguridad Ciudadana,Patrullaje policial en el sector solicitado,139469,1
152674,TUNGURAHUA,AMBATO,"AMBATO, CABECERA CANTONAL Y CAPITAL PROVINCIAL",Seguridad Ciudadana,Patrullaje policial en el sector solicitado,105095,1
44372,EL ORO,MACHALA,"MACHALA, CABECERA CANTONAL Y CAPITAL PROVINCIAL",Seguridad Ciudadana,Patrullaje policial en el sector solicitado,104439,1
33225,CHIMBORAZO,RIOBAMBA,"RIOBAMBA, CABECERA CANTONAL Y CAPITAL PROVINCIAL",Seguridad Ciudadana,Patrullaje policial en el sector solicitado,104142,1
83906,LOJA,LOJA,"LOJA, CABECERA CANTONAL Y CAPITAL PROVINCIAL",Seguridad Ciudadana,Patrullaje policial en el sector solicitado,97809,1
